In [1]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as patches
import matplotlib.cm as cm
import matplotlib as mpl

import scipy
import numpy as np
np.random.seed(42)
import ipywidgets as widgets

import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

In [2]:
SAMPLERATE = 250000
NUM_CHANNELS = 8
BYTES_PER_SAMPLE = 2 #int16

COLOR_MAP_FOR_TRAJ = {0:cm.Blues, 1:cm.Reds, 2:cm.Greens}
MIC_MARKER_THICKNESS = 2
POINT_SIZE = 75

In [3]:
GRID_SIZE = 25
CIRCLE_RADIUS = 10
SOUND_SPEED_AIR = 343
TIME_DURATION = 0.2
RCVR_COLORS = ['red', 'limegreen', 'blue', 'mediumpurple', 'slategray', 'brown', 'green', 'darkred']
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=RCVR_COLORS)
COLOR_CYCLE = plt.rcParams['axes.prop_cycle'].by_key()['color']
BAT_INIT_DIST = 5

FS = 250000
ASSUMED_BAT_SPEED = 4*np.sqrt(3)/3 # m/s
ASSUMED_BAT_IPI = 0.1 # secs (100ms)
TIMESTEPS = np.arange(0, ((2*BAT_INIT_DIST)/ASSUMED_BAT_SPEED) + ASSUMED_BAT_IPI, ASSUMED_BAT_IPI)

In [4]:
ZMAG_REF_TO_6 = 29.125
ZMAG_REF_TO_7 = 31.6875
XMAG_REF_TO_1 = 17.7
YMAG_REF_TO_6 = 21
YMAG_REF_TO_1 = 37.5
UBNA_ARRAY_MIC_LOCS =  (254/10000) * np.array([[-XMAG_REF_TO_1, -YMAG_REF_TO_1, ZMAG_REF_TO_7],
                    [-XMAG_REF_TO_1, YMAG_REF_TO_1, ZMAG_REF_TO_7],
                    [-XMAG_REF_TO_1, -YMAG_REF_TO_1, 0],
                    [-XMAG_REF_TO_1, YMAG_REF_TO_1, 0],
                    [-XMAG_REF_TO_1, -YMAG_REF_TO_6, -ZMAG_REF_TO_6],
                    [-XMAG_REF_TO_1, YMAG_REF_TO_6, -ZMAG_REF_TO_6],
                    [0, 0, ZMAG_REF_TO_7],
                    [0, 0, 0]])
UBNA_ARRAY_MIC_LOCS

array([[-0.44958  , -0.9525   ,  0.8048625],
       [-0.44958  ,  0.9525   ,  0.8048625],
       [-0.44958  , -0.9525   ,  0.       ],
       [-0.44958  ,  0.9525   ,  0.       ],
       [-0.44958  , -0.5334   , -0.739775 ],
       [-0.44958  ,  0.5334   , -0.739775 ],
       [ 0.       ,  0.       ,  0.8048625],
       [ 0.       ,  0.       ,  0.       ]])

In [5]:
%matplotlib inline

SELECTED_CHANNEL_FOR_REF = 7
MICROPHONES_USED = np.array([1,2,3,4,5,6,7,8])
IND_OF_SELECTED_CHANNEL = np.where(MICROPHONES_USED==(SELECTED_CHANNEL_FOR_REF+1))[0]
SELECTED_MIC_FOR_REF = MICROPHONES_USED[IND_OF_SELECTED_CHANNEL]
NON_REF_MICROPHONES_USED = MICROPHONES_USED[MICROPHONES_USED!=SELECTED_MIC_FOR_REF]
NUM_GOOD_CHANNELS = MICROPHONES_USED.shape[0]
NUM_NONREFCHANNELS = NUM_GOOD_CHANNELS-1

In [6]:
def simulate_N_receivers(time_step, noise_power_dB):
    N = UBNA_ARRAY_MIC_LOCS.shape[0]
    rN_x = UBNA_ARRAY_MIC_LOCS[:,0]
    rN_y = UBNA_ARRAY_MIC_LOCS[:,1]
    rN_z = UBNA_ARRAY_MIC_LOCS[:,2]
    
    single_time_steps = np.arange(0, (2*BAT_INIT_DIST/ASSUMED_BAT_SPEED)+ASSUMED_BAT_IPI, ASSUMED_BAT_IPI)
    forwards_xy = ASSUMED_BAT_SPEED * (single_time_steps) - (BAT_INIT_DIST)
    backwards_xy = - ASSUMED_BAT_SPEED * (single_time_steps) + (BAT_INIT_DIST)
    x_arr = np.concatenate([forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy])
    y_arr = np.repeat(np.arange(10, -11, -1), len(forwards_xy))
    z_arr = 10*np.ones(len(x_arr))
    pos_x = x_arr[time_step]
    pos_y = y_arr[time_step]
    pos_z = z_arr[time_step]

    fig = plt.figure(figsize=(18, 5))
    plt.rcParams.update({'font.size':12})
    gs = gridspec.GridSpec(1, 3, width_ratios=np.array([1, 1, 1]), wspace=0.3)
    map_ax_2d_xy = plt.subplot(gs[0, 0])
    map_ax_2d_xy.scatter(rN_x, rN_y, s=10, c=COLOR_CYCLE[:N])
    map_ax_2d_xy.plot(x_arr, y_arr, color='k', alpha=0.5, zorder=1, label=f"Source")
    map_ax_2d_xy.scatter(pos_x, pos_y, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    map_ax_2d_xy.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_xy.set_ylim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_xy.set_xlabel("X (meters)")
    map_ax_2d_xy.set_ylabel("Y (meters)")
    map_ax_2d_xy.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")

    map_ax_2d_xz = plt.subplot(gs[0, 1])
    map_ax_2d_xz.scatter(rN_x, rN_z, s=10, c=COLOR_CYCLE[:N])
    map_ax_2d_xz.plot(x_arr, z_arr, color='k', alpha=0.5, zorder=1, label=f"Source")
    map_ax_2d_xz.scatter(pos_x, pos_z, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    map_ax_2d_xz.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_xz.set_ylim(-GRID_SIZE, GRID_SIZE)
    map_ax_2d_xz.set_xlabel("X (meters)")
    map_ax_2d_xz.set_ylabel("Z (meters)")
    map_ax_2d_xz.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")

    map_ax_2d_yz = plt.subplot(gs[0, 2])
    map_ax_2d_yz.scatter(rN_y, rN_z, s=10, c=COLOR_CYCLE[:N])
    map_ax_2d_yz.plot(y_arr, z_arr, color='k', alpha=0.5, zorder=1, label=f"Source")
    map_ax_2d_yz.scatter(pos_y, pos_z, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    map_ax_2d_yz.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_yz.set_ylim(-GRID_SIZE, GRID_SIZE)
    map_ax_2d_yz.set_xlabel("Y (meters)")
    map_ax_2d_yz.set_ylabel("Z (meters)")
    map_ax_2d_yz.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")

    plt.show()

TIMESTEPS = np.arange(0, 21*((2*BAT_INIT_DIST/ASSUMED_BAT_SPEED)+ASSUMED_BAT_IPI), ASSUMED_BAT_IPI)
time_slider = widgets.IntSlider(
    value=len(TIMESTEPS)+7, min=0, max=len(TIMESTEPS)+7, step=1, description="Time (k)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

noise_slider = widgets.IntSlider(
    value=-10, min=-90, max=0, step=1, description="Noise Power (dBFS)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="800px")
)

interactive_plot = widgets.interactive(simulate_N_receivers, time_step=time_slider, noise_power_dB=noise_slider)
display(interactive_plot)

interactive(children=(IntSlider(value=938, description='Time (k)', layout=Layout(width='1000px'), max=938, sty…

### Simulation #1: Direct distance-to-array used

In [7]:
def simulate_N_receivers(time_step, noise_power_dB):
    N = UBNA_ARRAY_MIC_LOCS.shape[0]
    rN_x = UBNA_ARRAY_MIC_LOCS[:,0]
    rN_y = UBNA_ARRAY_MIC_LOCS[:,1]
    rN_z = UBNA_ARRAY_MIC_LOCS[:,2]
    A_LOCS_MAT = UBNA_ARRAY_MIC_LOCS
    A_locs_mat_wrt_ref_channel = A_LOCS_MAT
    A_locs_mat_tdoa_meters = A_locs_mat_wrt_ref_channel[(NON_REF_MICROPHONES_USED-1)]

    x0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][0]
    y0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][1]
    z0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][2]
    xm = A_locs_mat_tdoa_meters[:,0].reshape((NUM_NONREFCHANNELS, 1))
    ym = A_locs_mat_tdoa_meters[:,1].reshape((NUM_NONREFCHANNELS, 1))
    zm = A_locs_mat_tdoa_meters[:,2].reshape((NUM_NONREFCHANNELS, 1))
    
    single_time_steps = np.arange(0, (2*BAT_INIT_DIST/ASSUMED_BAT_SPEED)+ASSUMED_BAT_IPI, ASSUMED_BAT_IPI)
    forwards_xy = ASSUMED_BAT_SPEED * (single_time_steps) - (BAT_INIT_DIST)
    backwards_xy = - ASSUMED_BAT_SPEED * (single_time_steps) + (BAT_INIT_DIST)
    x_arr = np.concatenate([forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy])
    y_arr = np.repeat(np.arange(10, -11, -1), len(forwards_xy))
    z_arr = 10*np.ones(len(x_arr))
    pos_x = x_arr[time_step]
    pos_y = y_arr[time_step]
    pos_z = z_arr[time_step]
    
    dist_from_source_to_r_n_arr = np.sqrt((x_arr[None, :time_step] - rN_x[:, None]) ** 2 + (y_arr[None, :time_step] - rN_y[:, None]) ** 2 + (z_arr[None, :time_step] - rN_z[:, None]) ** 2)
    time_from_source_to_r_n_arr = (dist_from_source_to_r_n_arr[:,:] / SOUND_SPEED_AIR)
    true_time_delay_mics_to_selected_ref_channel = (time_from_source_to_r_n_arr - time_from_source_to_r_n_arr[IND_OF_SELECTED_CHANNEL,:]).T
    true_d_mics_to_ref = np.delete(true_time_delay_mics_to_selected_ref_channel * SOUND_SPEED_AIR, IND_OF_SELECTED_CHANNEL, axis=1)

    fig = plt.figure(figsize=(18, 5))
    plt.rcParams.update({'font.size':12})
    gs = gridspec.GridSpec(1, 3, width_ratios=np.array([1, 1, 1]), wspace=0.3)
    map_ax_2d_xy = plt.subplot(gs[0, 0])
    map_ax_2d_xy.scatter(rN_x, rN_y, s=10, c=COLOR_CYCLE[:N])
    map_ax_2d_xy.scatter(pos_x, pos_y, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    map_ax_2d_xy.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_xy.set_ylim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_xy.set_xlabel("X (meters)")
    map_ax_2d_xy.set_ylabel("Y (meters)")
    map_ax_2d_xy.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")

    map_ax_2d_xz = plt.subplot(gs[0, 1])
    map_ax_2d_xz.scatter(rN_x, rN_z, s=10, c=COLOR_CYCLE[:N])
    map_ax_2d_xz.scatter(pos_x, pos_z, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    map_ax_2d_xz.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_xz.set_ylim(-GRID_SIZE, GRID_SIZE)
    map_ax_2d_xz.set_xlabel("X (meters)")
    map_ax_2d_xz.set_ylabel("Z (meters)")
    map_ax_2d_xz.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")

    map_ax_2d_yz = plt.subplot(gs[0, 2])
    map_ax_2d_yz.scatter(rN_y, rN_z, s=10, c=COLOR_CYCLE[:N])
    map_ax_2d_yz.scatter(pos_y, pos_z, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    map_ax_2d_yz.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_yz.set_ylim(-GRID_SIZE, GRID_SIZE)
    map_ax_2d_yz.set_xlabel("Y (meters)")
    map_ax_2d_yz.set_ylabel("Z (meters)")
    map_ax_2d_yz.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")

    norm = plt.Normalize(vmin=0, vmax=TIMESTEPS.shape[0])
    source_locs = np.zeros((4, true_d_mics_to_ref.shape[0]-1), dtype=np.float64)
    cond_a_vals = np.zeros(true_d_mics_to_ref.shape[0]-1, dtype=np.float64)
    for i in range(true_d_mics_to_ref.shape[0]-1):
        dm0_column = true_d_mics_to_ref[i,:].reshape((NUM_NONREFCHANNELS, 1))
        A_mat = np.hstack(((A_locs_mat_tdoa_meters - A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF]), dm0_column))
        wm0 = ((xm**2 + ym**2 + zm**2) - (x0**2 + y0**2+ z0**2) - (dm0_column**2))/2

        xs, residuals, rank, sing_vals = scipy.linalg.lstsq(A_mat, wm0)
        source_locs[:,i] = xs.reshape((1,4))
        cond_a_vals[i] = np.linalg.cond(A_mat)
        
        line_color = COLOR_MAP_FOR_TRAJ[1](norm(i))
        map_ax_2d_xy.scatter(source_locs[0,i], source_locs[1,i], color=line_color, edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)
        map_ax_2d_xz.scatter(source_locs[0,i], source_locs[2,i], color=line_color, edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)
        map_ax_2d_yz.scatter(source_locs[1,i], source_locs[2,i], color=line_color, edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)

    dist_e = np.mean(np.sqrt(((source_locs[0,:]-x_arr[:time_step-1])**2) + ((source_locs[1,:]-y_arr[:time_step-1])**2) + ((source_locs[2,:]-z_arr[:time_step-1])**2)))
    map_ax_2d_xy.text(x=-10, y=11, s=f'Mean distance error: {dist_e:.6f}')
    
    plt.figure(figsize=(18, 2))
    plt.plot(cond_a_vals, marker='.')
    plt.grid(which='both')
    plt.ylabel('Cond(A)')
    plt.xlabel('Source location time step')
    plt.ylim(0, 300)

    plt.show()

TIMESTEPS = np.arange(0, 21*((2*BAT_INIT_DIST/ASSUMED_BAT_SPEED)+ASSUMED_BAT_IPI), ASSUMED_BAT_IPI)
time_slider = widgets.IntSlider(
    value=10, min=0, max=len(TIMESTEPS)+7, step=1, description="Time (k)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

noise_slider = widgets.IntSlider(
    value=-10, min=-90, max=0, step=1, description="Noise Power (dBFS)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="800px")
)

interactive_plot = widgets.interactive(simulate_N_receivers, time_step=time_slider, noise_power_dB=noise_slider)
display(interactive_plot)

interactive(children=(IntSlider(value=10, description='Time (k)', layout=Layout(width='1000px'), max=938, styl…

### Simulation #2: TDOA-based method for unit impulse transmission

In [9]:
def simulate_N_receivers(time_step, noise_power_dB):
    N = UBNA_ARRAY_MIC_LOCS.shape[0]
    rN_x = UBNA_ARRAY_MIC_LOCS[:,0]
    rN_y = UBNA_ARRAY_MIC_LOCS[:,1]
    rN_z = UBNA_ARRAY_MIC_LOCS[:,2]
    A_LOCS_MAT = UBNA_ARRAY_MIC_LOCS
    A_locs_mat_wrt_ref_channel = A_LOCS_MAT
    A_locs_mat_tdoa_meters = A_locs_mat_wrt_ref_channel[(NON_REF_MICROPHONES_USED-1)]

    x0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][0]
    y0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][1]
    z0 = A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF][2]
    xm = A_locs_mat_tdoa_meters[:,0].reshape((NUM_NONREFCHANNELS, 1))
    ym = A_locs_mat_tdoa_meters[:,1].reshape((NUM_NONREFCHANNELS, 1))
    zm = A_locs_mat_tdoa_meters[:,2].reshape((NUM_NONREFCHANNELS, 1))
    
    time_transmitted = TIME_DURATION/2
    t = np.linspace(0, TIME_DURATION, int(FS*TIME_DURATION))
    signal = scipy.signal.unit_impulse(t.size, idx=int(time_transmitted*FS)) * 1000
    signal_duration = TIME_DURATION/2

    single_time_steps = np.arange(0, (2*BAT_INIT_DIST/ASSUMED_BAT_SPEED)+ASSUMED_BAT_IPI, ASSUMED_BAT_IPI)
    forwards_xy = ASSUMED_BAT_SPEED * (single_time_steps) - (BAT_INIT_DIST)
    backwards_xy = - ASSUMED_BAT_SPEED * (single_time_steps) + (BAT_INIT_DIST)
    x_arr = np.concatenate([forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy, backwards_xy, forwards_xy, backwards_xy,
                            forwards_xy])
    y_arr = np.repeat(np.arange(10, -11, -1), len(forwards_xy))
    z_arr = 10*np.ones(len(x_arr))
    pos_x = x_arr[time_step]
    pos_y = y_arr[time_step]
    pos_z = z_arr[time_step]

    dist_from_source_to_r_n_arr = np.sqrt((x_arr[None, :time_step] - rN_x[:, None]) ** 2 + (y_arr[None, :time_step] - rN_y[:, None]) ** 2 + (z_arr[None, :time_step] - rN_z[:, None]) ** 2)
    time_to_rn_array = dist_from_source_to_r_n_arr / SOUND_SPEED_AIR
    receive_signaln_for_timestep = signal[:, None, None]
    signal_duration_samples = np.arange(int(signal_duration * FS))
    rn_indices = ((time_transmitted - time_to_rn_array) * FS).astype(int) + signal_duration_samples[:, None, None]
    chunk_of_received_signaln_in_window_timestep = np.take_along_axis(receive_signaln_for_timestep, rn_indices, axis=0)
    
    time_received_at_rn = ((np.argmax(chunk_of_received_signaln_in_window_timestep, axis=0)) / FS).T
    t_delay_ref_to_selected_channel = time_received_at_rn[:,IND_OF_SELECTED_CHANNEL]
    time_delay_mics_to_selected_ref_channel = (time_received_at_rn[:]) - (t_delay_ref_to_selected_channel.reshape((len(t_delay_ref_to_selected_channel), 1)))
    time_delay_mics_to_selected_ref_channel_no_ref = np.delete(time_delay_mics_to_selected_ref_channel, IND_OF_SELECTED_CHANNEL, axis=1)
    measured_d_mics_to_ref = (time_delay_mics_to_selected_ref_channel_no_ref * SOUND_SPEED_AIR)

    fig = plt.figure(figsize=(18, 5))
    plt.rcParams.update({'font.size':12})
    gs = gridspec.GridSpec(1, 3, width_ratios=np.array([1, 1, 1]), wspace=0.3)
    map_ax_2d_xy = plt.subplot(gs[0, 0])
    map_ax_2d_xy.scatter(rN_x, rN_y, s=10, c=COLOR_CYCLE[:N])
    map_ax_2d_xy.scatter(pos_x, pos_y, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    map_ax_2d_xy.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_xy.set_ylim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_xy.set_xlabel("X (meters)")
    map_ax_2d_xy.set_ylabel("Y (meters)")
    map_ax_2d_xy.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")

    map_ax_2d_xz = plt.subplot(gs[0, 1])
    map_ax_2d_xz.scatter(rN_x, rN_z, s=10, c=COLOR_CYCLE[:N])
    map_ax_2d_xz.scatter(pos_x, pos_z, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    map_ax_2d_xz.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_xz.set_ylim(-GRID_SIZE / 10, GRID_SIZE)
    map_ax_2d_xz.set_xlabel("X (meters)")
    map_ax_2d_xz.set_ylabel("Z (meters)")
    map_ax_2d_xz.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")

    map_ax_2d_yz = plt.subplot(gs[0, 2])
    map_ax_2d_yz.scatter(rN_y, rN_z, s=10, c=COLOR_CYCLE[:N])
    map_ax_2d_yz.scatter(pos_y, pos_z, edgecolors='red', facecolors='none', linewidths=2, s=50, label=f"Source")
    map_ax_2d_yz.set_xlim(-GRID_SIZE / 2, GRID_SIZE / 2)
    map_ax_2d_yz.set_ylim(-GRID_SIZE / 10, GRID_SIZE)
    map_ax_2d_yz.set_xlabel("Y (meters)")
    map_ax_2d_yz.set_ylabel("Z (meters)")
    map_ax_2d_yz.set_title(f"Sound Reception ({int(N)} receivers; 1 source)")

    norm = plt.Normalize(vmin=0, vmax=TIMESTEPS.shape[0])
    source_locs = np.zeros((4, measured_d_mics_to_ref.shape[0]-1), dtype=np.float64)
    cond_a_vals = np.zeros(measured_d_mics_to_ref.shape[0]-1, dtype=np.float64)
    for i in range(measured_d_mics_to_ref.shape[0]-1):
        dm0_column = measured_d_mics_to_ref[i,:].reshape((NUM_NONREFCHANNELS, 1))
        A_mat = np.hstack(((A_locs_mat_tdoa_meters - A_LOCS_MAT[SELECTED_CHANNEL_FOR_REF]), dm0_column))
        wm0 = ((xm**2 + ym**2 + zm**2) - (x0**2 + y0**2+ z0**2) - (dm0_column**2))/2

        xs, residuals, rank, sing_vals = scipy.linalg.lstsq(A_mat, wm0)
        source_locs[:,i] = xs.reshape((1,4))
        cond_a_vals[i] = np.linalg.cond(A_mat)
        line_color = COLOR_MAP_FOR_TRAJ[1](norm(i))
        map_ax_2d_xy.scatter(source_locs[0,i], source_locs[1,i], color=line_color, edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)
        map_ax_2d_xz.scatter(source_locs[0,i], source_locs[2,i], color=line_color, edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)
        map_ax_2d_yz.scatter(source_locs[1,i], source_locs[2,i], color=line_color, edgecolor='k', alpha=1, s=POINT_SIZE/2, zorder=2)

    dist_e = np.mean(np.sqrt(((source_locs[0,:]-x_arr[:time_step-1])**2) + ((source_locs[1,:]-y_arr[:time_step-1])**2) + ((source_locs[2,:]-z_arr[:time_step-1])**2)))
    map_ax_2d_xy.text(x=-10, y=11, s=f'Mean distance error: {dist_e:.6f}')
    
    plt.figure(figsize=(18, 2))
    plt.plot(cond_a_vals, marker='.')
    plt.grid(which='both')
    plt.ylabel('Cond(A)')
    plt.xlabel('Source location time step')
    plt.ylim(0, 300)

    plt.show()

TIMESTEPS = np.arange(0, 21*((2*BAT_INIT_DIST/ASSUMED_BAT_SPEED)+ASSUMED_BAT_IPI), ASSUMED_BAT_IPI)
time_slider = widgets.IntSlider(
    value=10, min=0, max=len(TIMESTEPS)+7, step=1, description="Time (k)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="1000px")
)

noise_slider = widgets.IntSlider(
    value=-10, min=-90, max=0, step=1, description="Noise Power (dBFS)", continuous_update=True, style={'description_width': 'initial'}, layout=widgets.Layout(width="800px")
)

interactive_plot = widgets.interactive(simulate_N_receivers, time_step=time_slider, noise_power_dB=noise_slider)
display(interactive_plot)

interactive(children=(IntSlider(value=10, description='Time (k)', layout=Layout(width='1000px'), max=938, styl…